# Progressive train-validation grid search

This notebook uses only the published train and validation manifests. It persists every trial immediately, skips completed configuration hashes on rerun, and never loads test data. Run it on CUDA after the CPU smoke test in `04_train_model.ipynb` has passed.

In [ ]:
from dataclasses import replace
from pathlib import Path
import json

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import numpy as np
import pandas as pd
import torch

from reservoir_segmentation.training.config import ExperimentConfig, RuntimeConfig
from reservoir_segmentation.training.data import (
    ChannelStatistics, build_split_datasets, fit_channel_statistics, write_channel_statistics,
)
from reservoir_segmentation.training.grid_search import (
    make_finalist_repetitions, make_progressive_baseline_configurations,
    make_weighted_configurations, run_grid_search,
)
from reservoir_segmentation.training.engine import model_from_checkpoint

def trial_table(records):
    columns = ['state', 'selection_assd_macro_px', 'validation_dice', 'best_epoch', 'loss_name', 'learning_rate', 'base_channels', 'batch_size', 'seed', 'alpha', 'sigma', 'error_type', 'error', 'trial_path']
    return pd.DataFrame(records).reindex(columns=columns).sort_values(
        ['state', 'selection_assd_macro_px'], na_position='last'
    )


In [ ]:
REPOSITORY_ROOT = next((path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (path / 'pyproject.toml').is_file()), None)
if REPOSITORY_ROOT is None:
    raise FileNotFoundError('Could not locate the repository root. Start Jupyter from the repository or its notebooks directory.')
GEOMETRY_ROOT = REPOSITORY_ROOT / 'data/processed/riverscope/geometry-v2'
SOURCE_PATH = REPOSITORY_ROOT / 'data/raw/RiverScope.zip'
ARTIFACT_ROOT = REPOSITORY_ROOT / 'artifacts/experiments/grid_search'
DEVICE = 'cuda'
RETRY_FAILED = False  # Set True only to rerun trials recorded as failed.

if DEVICE == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('Full grid search requires CUDA. Use CPU only for tests and smoke tests.')

raw_datasets = build_split_datasets(GEOMETRY_ROOT, SOURCE_PATH)
statistics = fit_channel_statistics(raw_datasets['train'])
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
write_channel_statistics(statistics, ARTIFACT_ROOT / 'train_channel_statistics.json')
datasets = build_split_datasets(GEOMETRY_ROOT, SOURCE_PATH, statistics=statistics)
statistics.to_dict()


In [ ]:
from reservoir_segmentation.training.data import build_dataloader

probe = build_dataloader(
    datasets["train"],
    batch_size=4,
    shuffle=False,
    seed=42,
    num_workers=2,
)
probe_batch = next(iter(probe))
probe_batch["image"].shape

## Baseline CE: 12 architecture/optimizer trials

Only learning rate, base channels, and batch size vary. AdamW, 100 maximum epochs, scheduler, early stopping, and the ASSD-with-Dice checkpoint rule remain fixed.

In [ ]:
template = ExperimentConfig(runtime=RuntimeConfig(seed=42, device=DEVICE, num_workers=2))
baseline_configs = make_progressive_baseline_configurations(template)
baseline_records = run_grid_search(
    baseline_configs, datasets['train'], datasets['valid'],
    output_dir=ARTIFACT_ROOT / 'baseline',
    channel_statistics=statistics.to_dict(),
    repository_root=REPOSITORY_ROOT,
    retry_failed=RETRY_FAILED,
)
trial_table(baseline_records)


In [ ]:
baseline_winner_payload = json.loads((ARTIFACT_ROOT / 'baseline/winner.json').read_text())
baseline_winner = ExperimentConfig.from_dict(baseline_winner_payload['experiment_config'])
baseline_finalists = run_grid_search(
    make_finalist_repetitions(baseline_winner), datasets['train'], datasets['valid'],
    output_dir=ARTIFACT_ROOT / 'baseline_finalists',
    channel_statistics=statistics.to_dict(), repository_root=REPOSITORY_ROOT,
    retry_failed=RETRY_FAILED,
)
trial_table(baseline_finalists)


## Weighted CE: 8 loss trials, fixed architecture

The selected CE architecture and batch size remain fixed. Only `alpha`, `sigma`, and the allowed finalist LR adjustment vary, keeping CE and weighted CE comparable.

In [ ]:
weighted_configs = make_weighted_configurations(baseline_winner)
weighted_records = run_grid_search(
    weighted_configs, datasets['train'], datasets['valid'],
    output_dir=ARTIFACT_ROOT / 'weighted',
    channel_statistics=statistics.to_dict(), repository_root=REPOSITORY_ROOT,
    retry_failed=RETRY_FAILED,
)
trial_table(weighted_records)


In [ ]:
weighted_winner_payload = json.loads((ARTIFACT_ROOT / 'weighted/winner.json').read_text())
weighted_winner = ExperimentConfig.from_dict(weighted_winner_payload['experiment_config'])
weighted_finalists = run_grid_search(
    make_finalist_repetitions(weighted_winner), datasets['train'], datasets['valid'],
    output_dir=ARTIFACT_ROOT / 'weighted_finalists',
    channel_statistics=statistics.to_dict(), repository_root=REPOSITORY_ROOT,
    retry_failed=RETRY_FAILED,
)
trial_table(weighted_finalists)


Review the two three-seed finalist tables using validation metrics only. Freeze a final configuration and checkpoint for `04_train_model.ipynb`; do not inspect or optimize on test data here.

## Validation reconstructions from the selected candidate

This diagnostic uses only validation tiles. It loads the best available finalist checkpoint from the CE and weighted-CE branches, selected by validation selection ASSD with Dice as a tiebreaker. It does not load or inspect test data.

In [ ]:
PREVIEW_CASES = 40

winner_paths = {
    name: ARTIFACT_ROOT / name / 'winner.json'
    for name in ('baseline_finalists', 'weighted_finalists')
}
winner_records = []
for experiment_name, winner_path in winner_paths.items():
    if winner_path.is_file():
        winner = json.loads(winner_path.read_text(encoding='utf-8'))
        winner['experiment_name'] = experiment_name
        winner_records.append(winner)
if not winner_records:
    raise FileNotFoundError('Run the finalist cells before rendering validation reconstructions.')

preview_winner = min(
    winner_records,
    key=lambda record: (
        float(record['selection_assd_macro_px']),
        -float(record['validation_dice']),
    ),
)
preview_checkpoint_path = Path(preview_winner['trial_path']) / 'best_checkpoint.pt'
preview_model, preview_config, preview_checkpoint = model_from_checkpoint(
    preview_checkpoint_path, device=DEVICE
)
preview_statistics = ChannelStatistics.from_dict(preview_checkpoint['channel_statistics'])
preview_dataset = build_split_datasets(
    GEOMETRY_ROOT, SOURCE_PATH, statistics=preview_statistics
)['valid']

def rgb_for_preview(normalized_image, valid_mask):
    mean = np.asarray(preview_statistics.mean)[:, None, None]
    std = np.asarray(preview_statistics.std)[:, None, None]
    rgb = np.moveaxis((normalized_image * std + mean)[:3], 0, -1)
    low, high = np.quantile(rgb[valid_mask], (0.02, 0.98))
    rgb = np.clip((rgb - low) / max(high - low, 1e-6), 0, 1)
    rgb[~valid_mask] = 0.5
    return rgb

rng = np.random.default_rng(43)
case_indices = sorted(rng.choice(len(preview_dataset), size=min(PREVIEW_CASES, len(preview_dataset)), replace=False))
fig, axes = plt.subplots(len(case_indices), 4, figsize=(16, 4 * len(case_indices)), squeeze=False)
error_cmap = ListedColormap(['black', 'lime', 'orange', 'red'])
for row_index, case_index in enumerate(case_indices):
    sample = preview_dataset[int(case_index)]
    image = sample['image'].numpy()
    target = sample['target'].numpy().astype(bool)
    valid = sample['valid_mask'].numpy().astype(bool)
    with torch.no_grad():
        prediction = preview_model.predict_labels(
            sample['image'].unsqueeze(0).to(DEVICE)
        )[0].cpu().numpy().astype(bool)
    errors = np.full(target.shape, np.nan)
    errors[valid & ~target & ~prediction] = 0
    errors[valid & target & prediction] = 1
    errors[valid & target & ~prediction] = 2
    errors[valid & ~target & prediction] = 3
    rgb = rgb_for_preview(image, valid)
    row = axes[row_index]
    row[0].imshow(rgb); row[0].set_title(f'{sample["tile_id"]}: RGB')
    row[1].imshow(np.ma.masked_where(~valid, target), cmap='Blues', vmin=0, vmax=1); row[1].set_title('Validation reference')
    row[2].imshow(np.ma.masked_where(~valid, prediction), cmap='Blues', vmin=0, vmax=1); row[2].set_title('Argmax reconstruction')
    row[3].imshow(errors, cmap=error_cmap, vmin=0, vmax=3); row[3].imshow(np.where(valid, np.nan, 1), cmap=ListedColormap(['gray']), alpha=0.75); row[3].set_title('TN / TP / FN / FP; gray invalid')
    for axis in row:
        axis.set_axis_off()
fig.suptitle(f"{preview_winner['experiment_name']} validation candidate: {preview_checkpoint_path}", fontsize=9)
fig.tight_layout()
preview_path = ARTIFACT_ROOT / 'validation_reconstructions.png'
fig.savefig(preview_path, dpi=160)
plt.show()
{'winner': preview_winner, 'visualization_path': str(preview_path)}
